# 07 — Consistency and Self-Consistency Prompting

In [1]:
import os, warnings
warnings.filterwarnings('ignore')
from dotenv import load_dotenv
load_dotenv(os.path.join(os.getcwd(), '..', '.env'))
GROQ_API_KEY = os.getenv('GROQ_API_KEY')
assert GROQ_API_KEY, "GROQ_API_KEY not found — check ../.env"
print("API key loaded")


API key loaded


In [2]:
from groq import Groq
client = Groq()
MODEL = "llama-3.1-8b-instant"
MODEL_LARGE = "llama-3.3-70b-versatile"

def chat(messages, model=MODEL, temperature=0.7, max_tokens=512):
    resp = client.chat.completions.create(
        model=model, messages=messages,
        temperature=temperature, max_tokens=max_tokens
    )
    return resp.choices[0].message.content

def system_chat(system, user, **kw):
    return chat([{"role": "system", "content": system},
                 {"role": "user", "content": user}], **kw)

print(f"Groq client ready | default model: {MODEL}")


Groq client ready | default model: llama-3.1-8b-instant


## Self-Consistency: Sample Multiple Paths, Vote

In [3]:
import re
from collections import Counter

problem = (
    "When I was 6 my sister was half my age. Now I am 70. How old is my sister?\n"
    "Think step by step, then give only the numeric answer on the last line."
)

answers = []
for i in range(5):
    r = chat([{"role":"user","content":problem}], temperature=0.8, max_tokens=150)
    lines = [l.strip() for l in r.strip().split("\n") if l.strip()]
    last  = lines[-1] if lines else ""
    nums  = re.findall(r"\d+", last)
    ans   = nums[-1] if nums else "?"
    answers.append(ans)
    print(f"Sample {i+1}: ...{r.strip()[-60:]}")
    print(f"  -> Answer: {ans}\n")

vote = Counter(answers).most_common(1)[0]
print(f"Self-consistency vote: {vote[0]} ({vote[1]}/5 agree)")


Sample 1: ...ter's age, subtract 3 from your current age: 70 - 3 = 67

67
  -> Answer: 67



Sample 2: ...tract the age difference from your current age: 70 - 3 = 67.
  -> Answer: 67



Sample 3: ...3 + 64 = 67 years old now.

So, your sister is 67 years old.
  -> Answer: 67



Sample 4: ...ve aged 70 - 6 = 64 years.

So, your sister has also aged 64
  -> Answer: 64



Sample 5: ...ge difference has been constant, your sister is 3 - 64 = -61
  -> Answer: 61

Self-consistency vote: 67 (3/5 agree)


## Consistency Check: Detecting Contradictions

In [4]:
q1 = "What is the boiling point of water at sea level in Celsius?"
q2 = "At standard atmospheric pressure, at what temperature does water turn to steam? (Celsius)"
q3 = "Water boils at __ degrees Celsius under normal conditions."

for q in [q1, q2, q3]:
    a = chat([{"role":"user","content":q}], temperature=0, max_tokens=30)
    print(f"Q: {q[:60]}")
    print(f"A: {a.strip()}")
    print()

answers_list = []
for q in [q1, q2, q3]:
    a = chat([{"role":"user","content":q}], temperature=0, max_tokens=30)
    answers_list.append(a.strip())

consistent = all("100" in a for a in answers_list)
print(f"Consistent (all contain 100): {consistent}")


Q: What is the boiling point of water at sea level in Celsius?
A: The boiling point of water at sea level is 100 degrees Celsius.



Q: At standard atmospheric pressure, at what temperature does w
A: At standard atmospheric pressure, water turns to steam at 100 degrees Celsius.



Q: Water boils at __ degrees Celsius under normal conditions.
A: Water boils at 100 degrees Celsius under normal atmospheric pressure conditions.



Consistent (all contain 100): True


## Majority Vote for Classification

In [5]:
from collections import Counter

text = "The product arrived damaged and customer support was unhelpful. Terrible experience."
prompt = f"Classify sentiment as POSITIVE, NEGATIVE, or NEUTRAL. Text: '{text}'. Answer in one word:"

votes = []
for _ in range(7):
    r = chat([{"role":"user","content":prompt}], temperature=0.9, max_tokens=10)
    word = r.strip().split()[0].upper().rstrip(".,!")
    if word in ("POSITIVE","NEGATIVE","NEUTRAL"):
        votes.append(word)
    else:
        votes.append("UNCLEAR")

cnt = Counter(votes)
print(f"Votes: {dict(cnt)}")
print(f"Final (majority): {cnt.most_common(1)[0][0]}")


Votes: {'NEGATIVE': 7}
Final (majority): NEGATIVE


## Confidence Elicitation

In [6]:
fact_q = "What year did the Berlin Wall fall?"
prompt = (
    f"{fact_q}\n\n"
    "Answer the question and then rate your confidence from 0-100%.\n"
    "Format:\n"
    "Answer: <your answer>\n"
    "Confidence: <0-100>%"
)
result = chat([{"role":"user","content":prompt}], temperature=0, max_tokens=60)
print(result.strip())


Answer: 1989
Confidence: 100%
